<a href="https://colab.research.google.com/github/Kovasc/Assistente-de-voz/blob/main/Assistente_de_Voz_Multi_Idiomas_Com_Whisper_e_ChatGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
language = 'pt'

# 1. Gravação de Áudio Com Python (e Uma Pitada de JavaScript) 🎤

In [14]:
# Referência: https://gist.github.com/korakot/c21c3476c024ad6d56d5f48b0bca92be

from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

# Código JavaScript para gravar áudio do usuário usando a "MediaStream Recording API"
RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record(sec=5):
  # Executa o código JavaScript para gravar o áudio
  display(Javascript(RECORD))
  # Recebe o áudio gravado como resultado do JavaScript
  js_result = output.eval_js('record(%s)' % (sec * 1000))
   # Decodifica o áudio em base64
  audio = b64decode(js_result.split(',')[1])
  # Salva o áudio em um arquivo
  file_name = 'request_audio.wav'
  with open(file_name, 'wb') as f:
    f.write(audio)
  # Retorna o caminho do arquivo de áudio (pasta padrão do Google Colab)
  return f'/content/{file_name}'

# Grava o áudio do usuário por um tempo determinado (padrão 5 segundos)
print('Ouvindo...\n')
record_file = record()

# Exibe o áudio gravado
display(Audio(record_file, autoplay=False))

Ouvindo...



<IPython.core.display.Javascript object>

# 2. Reconhecimento de Fala com Whisper (OpenAI) 🧠

In [3]:
!pip install git+https://github.com/openai/whisper.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 4.6 MB/s eta 0:00:00


In [15]:
import whisper

# Selecione o modelo do Whisper que melhor atenda às suas necessidades:
# https://github.com/openai/whisper#available-models-and-languages
model = whisper.load_model("medium")

# Transcreve o audio gravado anteriormente.
result = model.transcribe(record_file, fp16=False, language=language, task="translate")
transcription = result["text"]
print(transcription)

 So chat tell me how is the weather today here in the city of Anápolis


# 3. Integração com a API do ChatGPT 💬

In [27]:
!pip install openai

In [19]:
from openai import OpenAI
from google.colab import userdata

api_key = userdata.get('OPENAI_API_KEY')

client = OpenAI(
  api_key=api_key
)

response = client.responses.create(
  model="gpt-5-nano",
  input=[ { "role": "user", "content": transcription } ],
  store=True,
)

print(response.output_text);

I can’t fetch real-time weather data from here. But I can help you get it quickly or give you a quick climate overview for Anápolis.

Ways to check current conditions:
- Google: search “Anápolis weather today” (instant update).
- Your weather app: set location to Anápolis, GO, Brazil.
- Websites: Weather.com, AccuWeather, or INMET (inmet.gov.br) for official data.

What I can share without live data:
- Anápolis is in Goiás, Brazil, with a tropical savanna climate. It’s hot most of the year; the rainy season is roughly October through April, and the dry season is May through September. Typical daytime highs are around 30–35°C (86–95°F), with cooler nights.
- If you tell me the forecast you’re seeing (temperature, rain chance, wind), I can help interpret it.

Would you like me to guide you through checking it now, or provide more detailed climate info?


# 4. Sintetizando a Resposta do ChatGPT Como Voz (gTTS) 🔊

In [1]:
!pip install gTTS

In [17]:
from gtts import  gTTS

# Cria um objeto gTTS com a resposta gerada pelo ChatGPT e a língua que será sintetizada em voz (variável "language").
gtts_object = gTTS(text=response.output_text, slow=False)

# Salva o áudio da resposta no arquivo especificado (pasta padrão do Google Colab)
response_audio = "/content/response_audio.wav"
gtts_object.save(response_audio)

# Reproduz o áudio da resposta salvo no arquivo
display(Audio(response_audio, autoplay=True))